In [1]:
### importing the required libraries
using CSV
using DataFrames
using Statistics
using Plots
using DataStructures  # For the Counter function
using Tar
using CodecZlib

In [2]:
rel_propn = 1
## vary these for different dispersal scenarios
gamma = 0.33    # dispersal rate from release site to household
rho = 0.33     # rate visit the households
tau = 0.33      # rate leave the households
alpha = 0.1     # dispersal rate of females moving from household directly to another household

rel_fem = 5
rel_male = 5
delta_rel = 10

num_simulations = 100  # number of realisations of SSA to run, 10,000 is a good number for the paper
rel_type = "relhhold"

"relhhold"

In [3]:
# Define parameters dictionary
fem_w0 = 0  # how many wildtypes and infected mosquitoes starting off with
male_w0 = 0
fem_m0 = 8
male_m0 = 8
m_free0 = 0 # no. of free wild-type
w_free0 = 0 # no. of free Wolbachia

# model parameters
phi = 0.85      # wolbachia fitness effect on birth rate
K = 30          # carrying capacity of mosquitoes per household
d = 12/100      # death rate of mosquitoes
k = 0.3         # larval density parameter
h = 0.19*100^k  # larval density parameter
b = 0.54        # per capita birth rate of mosquitoes (wildtypes)
H = 100         # number of households
u = 1           # vertical transmission probability
v = 1           # CI effect, proportion of non-viable offspring from infected male and wildtype female birth

rel_t = 150   # initial release time

## remember this is per household for the household releases so should use smaller release numbers
## currently set for community wide release

t_start = 0    
t_end = 1000   # start time and end time (days) of simulation

result_length = length(t_start:t_end) # number of time points to store results
weeks = round(Int,result_length/7) + 1   # number of weeks to store results

parameters = Dict(           # dictionary of parameters
    :fem_m0 => fem_m0,
    :male_m0 => male_m0,
    :fem_w0 => fem_w0,
    :male_w0 => male_w0,
    :m_free0 => m_free0,
    :w_free0 => w_free0,
    :rho => rho,
    :phi => phi,
    :b => b,
    :K => K,
    :d => d,
    :h => h,
    :k => k,
    :u => u,
    :v => v,
    :tau => tau,
    :H => H,
    :t_start => t_start,
    :t_end => t_end,
    :seed => 1234,
    :delta_rel => delta_rel,
    #:rel_size => rel_size,
    :gamma => gamma,
    :alpha => alpha,
    :rel_t => rel_t
)

Dict{Symbol, Real} with 24 entries:
  :b       => 0.54
  :alpha   => 0.1
  :gamma   => 0.33
  :m_free0 => 0
  :rho     => 0.33
  :t_start => 0
  :h       => 0.756404
  :rel_t   => 150
  :male_w0 => 0
  :fem_m0  => 8
  :K       => 30
  :phi     => 0.85
  :fem_w0  => 0
  :d       => 0.12
  :k       => 0.3
  :v       => 1
  :u       => 1
  :male_m0 => 8
  :tau     => 0.33
  ⋮        => ⋮

In [4]:
### Convert dispersal parameters to a string and remove the decimal point
## We will use this to call the correct data files
rho_str = replace(string(rho), "." => "")
tau_str = replace(string(tau), "." => "")
gamma_str = replace(string(gamma), "." => "")
alpha_str = replace(string(alpha), "." => "")

"01"

Function to find job ID

In [5]:
# Read the CSV file into a DataFrame
JOBS_DF = CSV.read("reg_rel_all_hholds.csv", DataFrame)

# Function to find the JOB_ID for given rel_fem and rel_male
function find_job_id(df::DataFrame, rel_fem_value, rel_male_value, rel_time)
    found = false
    for row in eachrow(df)
        if !ismissing(row[Symbol("job no.")]) &&
           row[Symbol("no. females")] == rel_fem_value && row[Symbol("no. males")] == rel_male_value &&
           row[Symbol("release time step")] == rel_time
            return row[Symbol("job no.")]
            found = true
            break
        else
            continue
        end
    end # Return nothing if no matching row is found
    if !found
        return nothing
    end
end

find_job_id (generic function with 1 method)

In [6]:
# Extract the .tar.gz file
function extract_tar_gz(file_path::String, dest_dir::String)
    open(file_path) do io
        tar_io = GzipDecompressorStream(io)
        Tar.extract(tar_io, dest_dir)
    end
end

# Define the sex ratios and release time steps
sex_ratios = [(1,1), (3,1), (3,3), (3,5), (3,10), (5,1), (5,3), (5,5), (5,10), (10,3), (10,5), (10,10), (10,1)]
rel_time_steps = [150, 200, 250, 300, 350]

for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        # Define the file path
        file_path = "all_hholds_release\\sim_results_$(JOB_ID).tar.gz" #testing_gen_hhold_release\\
        # Define the destination directory
        dest_dir =  joinpath("all_hholds_release", "$(JOB_ID)")  #joinpath("testing_gen_hhold_release",)
        # Extract the files
        extract_tar_gz(file_path, dest_dir)
        println("Extraction complete. Files are extracted to $dest_dir.")
    end 
end

1  1  150  67148348
Extraction complete. Files are extracted to all_hholds_release\67148348.
1  1  200  67148349
Extraction complete. Files are extracted to all_hholds_release\67148349.
1  1  250  67148350
Extraction complete. Files are extracted to all_hholds_release\67148350.
1  1  300  67148351
Extraction complete. Files are extracted to all_hholds_release\67148351.
1  1  350  67148352
Extraction complete. Files are extracted to all_hholds_release\67148352.
3  1  150  67148373
Extraction complete. Files are extracted to all_hholds_release\67148373.
3  1  200  67148374
Extraction complete. Files are extracted to all_hholds_release\67148374.
3  1  250  67148375
Extraction complete. Files are extracted to all_hholds_release\67148375.
3  1  300  67148376
Extraction complete. Files are extracted to all_hholds_release\67148376.
3  1  350  67148377
Extraction complete. Files are extracted to all_hholds_release\67148377.
3  3  150  67148356
Extraction complete. Files are extracted to all_hh

Invasion probability across non-releasing households

In [6]:
# Define the sex ratios and release time steps
sex_ratios = [(1,1), (3,1), (3,3), (3,5), (3,10), (5,1), (5,3), (5,5), (5,10), (10,3), (10,5), (10,10), (10,1)]
rel_time_steps = [150, 200, 250, 300, 350]

5-element Vector{Int64}:
 150
 200
 250
 300
 350

In [7]:
# Function to find the last non-empty column indices containing ones for each row in a matrix
function find_columns_with_ones(matrix::Matrix)
    result = []
    for row in eachrow(matrix)
        cols_with_ones = findall(x -> x == 1, row)  # Find indices where the value is 1
        push!(result, cols_with_ones)
    end

    # Find the last non-empty column indices
    for i in reverse(1:length(result))
        if !isempty(result[i])
            return result[i], i  # Return the last non-empty column indices and row index
        end
    end

    # Return empty if no ones are found
    return [], nothing
end

function count_columns_with_zero(df::Matrix)
    count_zeros = 0
    for col in eachcol(df)
        if 0.0 in col
            count_zeros += 1
        end
    end
    return count_zeros
end

# Convert symbols to integers in a nested array
function convert_symbols_to_integers(nested_array)
    return [map(x -> parse(Int, String(x)), inner_array) for inner_array in nested_array]
end

# Initialize the dictionary to store ratios
wild_av_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()
wolb_av_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()
wild_free_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

# Loop through each combination of sex ratios and release time steps
for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        wild_av = 0
        wolb_av = 0
        wolb_free_count = 0
        for i in 1:num_simulations
            df_does_rel = CSV.read(joinpath("all_hholds_release","$(JOB_ID)", "$(JOB_ID)", "does_rel_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_does_rel = Matrix(df_does_rel)
            df_fem_m_hhold = CSV.read(joinpath("all_hholds_release","$(JOB_ID)", "$(JOB_ID)", "fem_m_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_male_m_hhold = CSV.read(joinpath("all_hholds_release","$(JOB_ID)", "$(JOB_ID)", "fem_m_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_m_hhold = df_fem_m_hhold .+ df_male_m_hhold
            df_fem_w_hhold = CSV.read(joinpath("all_hholds_release","$(JOB_ID)", "$(JOB_ID)", "fem_w_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_male_w_hhold = CSV.read(joinpath("all_hholds_release","$(JOB_ID)", "$(JOB_ID)", "male_w_hholds_$(i)_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
            df_w_hhold = df_fem_w_hhold .+ df_male_w_hhold
            ## need to decide whether want end values or before dies out
            #columns_with_ones, len = find_columns_with_ones(df_does_rel)
            #println("Columns with ones: ", columns_with_ones, "  ", len)
            filtered_array = df_m_hhold[end,:]#[len, columns_with_ones]
            filtered_array = reshape(collect(filtered_array), 1, :)
            #println("Filtered array: ", filtered_array)
            #println(count_columns_with_zero(filtered_array)/size(filtered_array, 2))
            wolb_free_count += count_columns_with_zero(filtered_array)/size(filtered_array, 2)
            wild_av += mean(df_m_hhold[end, :])  # average wildtype household size
            #print(df_w_hhold[end, columns_with_ones])
            wolb_av += mean(df_w_hhold[end, :])  # average Wolbachia household size
        end
        # Store the ratio in the dictionary
        wild_av_dict[(r, t)] = wild_av/num_simulations
        wolb_av_dict[(r, t)] = wolb_av/num_simulations     
        wild_free_dict[(r, t)] = wolb_free_count/num_simulations
    end
end

1  1  150  67148348
1  1  200  67148349
1  1  250  67148350
1  1  300  67148351
1  1  350  67148352
3  1  150  67148373
3  1  200  67148374
3  1  250  67148375
3  1  300  67148376
3  1  350  67148377
3  3  150  67148356
3  3  200  67148357
3  3  250  67148358
3  3  300  67148359
3  3  350  67148360
3  5  150  67148423
3  5  200  67148424
3  5  250  67148425
3  5  300  67148426
3  5  350  67148427
3  10  150  67148451
3  10  200  67148452
3  10  250  67148453
3  10  300  67148454
3  10  350  67148455
5  1  150  67148386
5  1  200  67148387
5  1  250  67148388
5  1  300  67148389
5  1  350  67148390
5  3  150  67148470
5  3  200  67148471
5  3  250  67148472
5  3  300  67148473
5  3  350  67148474
5  5  150  67148340
5  5  200  67148341
5  5  250  67148342
5  5  300  67148343
5  5  350  67148344
5  10  150  67148460
5  10  200  67148461
5  10  250  67148462
5  10  300  67148463
5  10  350  67148464
10  3  150  67148694
10  3  200  67148695
10  3  250  67148696
10  3  300  67148697
10  3 

Proportion of wildtype-free households at time 1000 days (averaging over individual households)

In [8]:
# Convert the dictionary to a DataFrame
sex_ratios_col = [string(k[1]) for k in keys(wild_free_dict)]
release_time_steps_col = [k[2] for k in keys(wild_free_dict)]
wild_free_col = [v for v in values(wild_free_dict)]

wild_free_df = DataFrame(SexRatio=sex_ratios_col, ReleaseTimeStep=release_time_steps_col, InvProb=wild_free_col)
# Save the DataFrame as a CSV file
CSV.write("wild_free_all_hholds_rel.csv", wild_free_df)
# Load the CSV file into a DataFrame
#wolb_pers_df_loaded = CSV.read("wolb_pers_propn_hhold_10.csv", DataFrame)
wild_free_dict

Dict{Tuple{Tuple{Int64, Int64}, Int64}, Float64} with 65 entries:
  ((3, 3), 300)   => 1.0
  ((5, 1), 250)   => 1.0
  ((5, 10), 200)  => 1.0
  ((5, 5), 350)   => 1.0
  ((5, 10), 250)  => 1.0
  ((5, 5), 150)   => 1.0
  ((10, 1), 350)  => 1.0
  ((10, 1), 150)  => 1.0
  ((1, 1), 300)   => 0.9133
  ((10, 10), 350) => 1.0
  ((5, 3), 300)   => 1.0
  ((10, 10), 150) => 1.0
  ((3, 1), 300)   => 1.0
  ((10, 5), 300)  => 1.0
  ((3, 10), 300)  => 1.0
  ((3, 5), 200)   => 1.0
  ((10, 3), 200)  => 1.0
  ((3, 5), 250)   => 1.0
  ((10, 3), 250)  => 1.0
  ⋮               => ⋮

Average Wolbachia household size

In [9]:
# Convert the dictionary to a DataFrame
sex_ratios_col = [string(k[1]) for k in keys(wolb_av_dict)]
release_time_steps_col = [k[2] for k in keys(wolb_av_dict)]
wolb_col = [v for v in values(wolb_av_dict)]

inv_df = DataFrame(SexRatio=sex_ratios_col, ReleaseTimeStep=release_time_steps_col, WolbAv=wolb_col)
# Save the DataFrame as a CSV file
CSV.write("wolb_av_hhold_all_rel.csv", inv_df)
# Load the CSV file into a DataFrame
#wolb_pers_df_loaded = CSV.read("wolb_pers_propn_hhold_10.csv", DataFrame)
wolb_av_dict

Dict{Tuple{Tuple{Int64, Int64}, Int64}, Float64} with 65 entries:
  ((3, 3), 300)   => 9.7746
  ((5, 1), 250)   => 10.421
  ((5, 10), 200)  => 11.4878
  ((5, 5), 350)   => 10.1029
  ((5, 10), 250)  => 10.4304
  ((5, 5), 150)   => 10.4869
  ((10, 1), 350)  => 10.1191
  ((10, 1), 150)  => 10.6207
  ((1, 1), 300)   => 6.7501
  ((10, 10), 350) => 10.0153
  ((5, 3), 300)   => 9.8369
  ((10, 10), 150) => 10.5665
  ((3, 1), 300)   => 9.4075
  ((10, 5), 300)  => 9.9117
  ((3, 10), 300)  => 9.9619
  ((3, 5), 200)   => 11.0183
  ((10, 3), 200)  => 12.1114
  ((3, 5), 250)   => 10.325
  ((10, 3), 250)  => 10.4681
  ⋮               => ⋮

Persistence time of Wolbachia

In non-releasing households.

In [11]:
wolb_pers_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        #println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        df_track_fem_w = CSV.read(joinpath("all_hholds_release", "$(JOB_ID)", "$(JOB_ID)","track_fem_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
        df_track_male_w = CSV.read(joinpath("all_hholds_release", "$(JOB_ID)", "$(JOB_ID)","track_male_w_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)

        wolb_persist = df_track_fem_w .+ df_track_male_w # sum males and females
        n_rows, n_cols = size(wolb_persist)
        indices_zero_after_non_zero = fill(t_end+1, n_cols)  # Initialize with t_end to indicate no such zero found
        indices_first_non_zero = fill(t_end+1, n_cols)      # Initialize with t_end to indicate no non-zero found

        for col in 1:n_cols  # finds index where wolbachia first introduced and index where it goes extinct (if does)
            found_non_zero = false
            for row in 1:n_rows
                if wolb_persist[row, col] != 0
                    if !found_non_zero
                        indices_first_non_zero[col] = row
                        found_non_zero = true
                    end
                elseif found_non_zero && wolb_persist[row, col] == 0
                    indices_zero_after_non_zero[col] = row
                    break
                end
            end
        end

        wolb_pers_dict[(r, t)] = mean(indices_zero_after_non_zero - indices_first_non_zero)
    end
end

# Convert the dictionary to a DataFrame
sex_ratios_col = [string(k[1]) for k in keys(wolb_pers_dict)]
release_time_steps_col = [k[2] for k in keys(wolb_pers_dict)]
pers_time_col = [v for v in values(wolb_pers_dict)]

println(wolb_pers_dict)

wolb_pers_df = DataFrame(SexRatio=sex_ratios_col, ReleaseTimeStep=release_time_steps_col, PersTime=pers_time_col)

# Save the DataFrame as a CSV file
CSV.write("wolb_pers_all_hholds_norel.csv", wolb_pers_df)
# Load the CSV file into a DataFrame
#wolb_pers_df_loaded = CSV.read("wolb_pers_propn_hhold_10.csv", DataFrame)

ArgumentError: ArgumentError: "all_hholds_release\67148348\67148348\track_fem_w_100__150_1_1_150_relhhold.csv" is not a valid file or doesn't exist

In releasing households

In [7]:
wolb_pers_dict = Dict{Tuple{Tuple{Int, Int}, Int}, Float64}()

for r in sex_ratios
    for t in rel_time_steps
        fem, male = r
        JOB_ID = find_job_id(JOBS_DF, fem, male, t)
        #println(fem, "  ", male, "  ", t, "  ", JOB_ID)
        df_track_fem_w = CSV.read(joinpath("all_hholds_release", "$(JOB_ID)", "$(JOB_ID)","track_fem_w_rel_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)
        df_track_male_w = CSV.read(joinpath("all_hholds_release", "$(JOB_ID)", "$(JOB_ID)","track_male_w_rel_$(H)__$(rel_t)_$(male)_$(fem)_$(t)_$(rel_type).csv"), DataFrame)

        wolb_persist = df_track_fem_w .+ df_track_male_w # sum males and females
        n_rows, n_cols = size(wolb_persist)
        indices_zero_after_non_zero = fill(t_end+1, n_cols)  # Initialize with t_end to indicate no such zero found
        indices_first_non_zero = fill(t_end+1, n_cols)      # Initialize with t_end to indicate no non-zero found

        for col in 1:n_cols  # finds index where wolbachia first introduced and index where it goes extinct (if does)
            found_non_zero = false
            for row in 1:n_rows
                if wolb_persist[row, col] != 0
                    if !found_non_zero
                        indices_first_non_zero[col] = row
                        found_non_zero = true
                    end
                elseif found_non_zero && wolb_persist[row, col] == 0
                    indices_zero_after_non_zero[col] = row
                    break
                end
            end
        end

        wolb_pers_dict[(r, t)] = mean(indices_zero_after_non_zero - indices_first_non_zero)
    end
end

# Convert the dictionary to a DataFrame
sex_ratios_col = [string(k[1]) for k in keys(wolb_pers_dict)]
release_time_steps_col = [k[2] for k in keys(wolb_pers_dict)]
pers_time_col = [v for v in values(wolb_pers_dict)]

println(wolb_pers_dict)

wolb_pers_df = DataFrame(SexRatio=sex_ratios_col, ReleaseTimeStep=release_time_steps_col, PersTime=pers_time_col)

# Save the DataFrame as a CSV file
CSV.write("wolb_pers_all_hholds.csv", wolb_pers_df)
# Load the CSV file into a DataFrame
#wolb_pers_df_loaded = CSV.read("wolb_pers_propn_hhold_10.csv", DataFrame)

Dict(((3, 3), 300) => 183.49, ((5, 1), 250) => 442.08, ((5, 10), 200) => 624.81, ((5, 5), 350) => 521.32, ((5, 10), 250) => 626.4, ((5, 5), 150) => 545.16, ((10, 1), 350) => 648.9, ((10, 1), 150) => 650.95, ((1, 1), 300) => 13.13, ((10, 10), 350) => 697.98, ((5, 3), 300) => 520.29, ((10, 10), 150) => 752.53, ((3, 1), 300) => 172.55, ((10, 5), 300) => 629.23, ((3, 10), 300) => 376.93, ((3, 5), 200) => 342.46, ((10, 3), 200) => 700.55, ((3, 5), 250) => 219.36, ((10, 3), 250) => 705.92, ((3, 3), 350) => 193.78, ((5, 1), 300) => 499.36, ((3, 3), 150) => 243.97, ((5, 10), 300) => 586.54, ((5, 5), 200) => 543.18, ((5, 5), 250) => 541.77, ((1, 1), 350) => 12.99, ((10, 1), 200) => 627.53, ((5, 3), 350) => 480.92, ((1, 1), 150) => 12.5, ((10, 1), 250) => 669.13, ((10, 10), 200) => 703.0, ((5, 3), 150) => 509.23, ((3, 1), 350) => 137.0, ((10, 5), 350) => 649.37, ((10, 10), 250) => 693.71, ((3, 1), 150) => 226.14, ((10, 5), 150) => 696.1, ((3, 10), 350) => 436.92, ((3, 10), 150) => 388.43, ((3, 5

"wolb_pers_all_hholds.csv"